# EOG: dynamic research, forecast and DCF

No financial statement values or fixed year lists are embedded in notebook code.

`FINANCIAL_SOURCE = "wrds"` reads your latest saved WRDS report. `"sec"` reads SEC standard XBRL facts, with an optional online refresh. The chosen source stays explicit because its definitions differ from the other source. The current setting uses the saved SEC download and compares it with your WRDS cache. Cached runs do not prompt for credentials.

The operating data are previously reviewed observations stored separately by ticker, year, metric, units, and source. They are **not automatically refreshed** in that earlier table. Section 9 independently retrieves the six production/price inputs and revenue components from supported annual filing layouts. Unavailable years stay blank, so old values can never be relabeled as a new fiscal year.

Sections 12-16 add the forward model: a sourced assumptions table, the FY2026E-FY2031E operating forecast and unlevered free cash flow, WACC, both terminal values, the enterprise-to-equity bridge, scenarios and sensitivities. Forward choices live in `data/assumptions/EOG_assumptions.json`; filed guidance lives in `data/reference/eog_guidance.json`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (current, *current.parents) if (p / "app/history.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the DCF-Model project folder.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Reload edited project modules before rebuilding the annual DataFrames.
# Jupyter otherwise keeps previously imported Python code in memory.
import importlib
import app.sec as sec_module
import app.history as history_module
import app.history_review as review_module

if Path(history_module.__file__).resolve() != PROJECT_ROOT / "app/history.py":
    raise RuntimeError("A different app.history is loaded. Restart the kernel from this project and Run All.")

importlib.reload(sec_module)
importlib.reload(history_module)
importlib.reload(review_module)

from app.history import load_history, load_operating_observations, HistoryError
from app.sec import SECError

TICKER = "EOG"
HISTORY_YEARS = 5
END_YEAR = None                # None = newest annual period in the selected source
FINANCIAL_SOURCE = "sec"       # Audited filing data; WRDS stays available for comparison
SEC_CIK = "0000821189"         # Company identifier; change with ticker for SEC
REFRESH_SEC = False           # Latest SEC download is cached; True requests a new snapshot

history = load_history(
    ticker=TICKER, years=HISTORY_YEARS, end_year=END_YEAR,
    provider=FINANCIAL_SOURCE, cik=SEC_CIK,
    refresh=REFRESH_SEC if FINANCIAL_SOURCE == "sec" else False,
    root=PROJECT_ROOT,
)
annual = history.annual
print(f"{TICKER}: {annual.index.min()} through {annual.index.max()}")
print(f"Provider: {history.provider} | retrieved UTC: {history.retrieved_at}")
print(f"Raw source: {history.raw_path.relative_to(PROJECT_ROOT)}")
for message in history.warnings:
    print("NOTE:", message)


## 1. Standardized financial history

Amounts are USD millions, except diluted weighted-average shares in millions. Names describe a common schema; the audit table records the actual provider field. EBITDA is operating income plus the selected provider's D&A. Compustat revenue, D&A, debt, and cash can have different definitions from the filed statement lines.

The expanded WRDS refresh requests balance-sheet and tax inputs. The coverage and reconciliation sections below show whether they are present in the selected report. Missing fields remain blank.
For EOG, SEC capex is oil-and-gas property **and equipment** additions plus other PP&E additions. Mineral-interest purchases alone are not the total. Where SEC does not tag total liabilities, the EOG adapter sums current liabilities, noncurrent debt, other noncurrent liabilities, and deferred tax liabilities. The audit table shows these formulas and their component sources.

The existing `capex` field sums the property-addition lines before EOG's separately reported investing working-capital adjustment. Section 11 shows that adjustment and the complete investing cash-flow reconciliation.


In [ ]:
financial_fields = [
    "revenue", "operating_expenses", "operating_expenses_before_da",
    "ebit", "dd_a", "ebitda", "interest_expense", "pretax_income",
    "income_tax", "net_income", "operating_cash_flow", "capex",
    "cash", "receivables", "inventory", "accounts_payable",
    "debt_current", "debt_noncurrent", "total_debt", "diluted_shares",
]
history_table = annual[financial_fields].T
history_table.index.name = "Standard field"
display(history_table.round(2))

## 2. Input coverage and source audit

This shows gaps before calculations. Each retrieved value has an original field/tag, reporting period, raw value, unit scale, retrieval time, and filing reference where available. Calculated entries identify their component fields. SEC selection uses actual period dates and the latest available annual disclosure; this is not a point-in-time backtest. Company custom XBRL tags need a separate mapping and are left unavailable.

In [ ]:
coverage = pd.DataFrame({
    "Available years": annual[financial_fields].notna().sum(),
    "Requested years": len(annual),
})
coverage["Missing years"] = [
    ", ".join(map(str, annual.index[annual[field].isna()]))
    for field in financial_fields
]
display(coverage)

audit_columns = [
    "year", "metric", "value", "units", "provider", "source_field",
    "original_value", "original_units", "period_start", "period_end",
    "filed", "retrieved_at_utc", "source", "status",
]
display(history.audit.reindex(columns=audit_columns))

## 2a. Accounting reconciliation

These checks compare related fields in the selected source for every displayed year. Residuals are in USD millions. A missing input cannot pass; a difference stays visible for review and is never automatically adjusted. The default tolerance is $0.05 million.

A tie confirms internal consistency. It does not establish that Compustat definitions match EOG's filing lines. Discontinued operations or noncontrolling interests may explain differences in these simplified bridges for other companies or periods. Debt scope and operating working capital still require source review.


In [ ]:
from app.history_review import accounting_checks

reconciliation_checks = accounting_checks(history)
display(reconciliation_checks)
print(reconciliation_checks["Status"].value_counts().to_string())

key_inputs = [
    "receivables", "inventory", "accounts_payable", "pretax_income",
    "income_tax", "debt_current", "debt_noncurrent", "change_nwc_proxy",
]
input_review = pd.DataFrame({
    "Years available": annual[key_inputs].notna().sum(),
    "Years requested": len(annual),
})
display(input_review)
print("NWC remains a balance-sheet proxy: total receivables + inventory - accounts payable.")
print("Review non-operating balances, acquisition effects and capital payables before UFCF.")


## 3. Financial ratios, debt and working-capital diagnostics

`total_debt` is the sum of the selected provider's current and noncurrent debt. It is not automatically a valuation debt balance. In the saved EOG extract, Compustat includes operating leases in these fields; do not add them again. SEC debt and lease tags are displayed separately for reconciliation.

The working-capital proxy is **receivables + inventory - accounts payable**. A positive annual change is an increased balance-sheet investment. Acquisition balances, joint-interest accounts, capital payables, and omitted current accounts still require review before treating this as DCF operating NWC. The loader retains one preceding year for changes; missing prior-year inputs stay blank.

In [ ]:
ratio_fields = [
    "revenue_growth_pct", "ebit_margin_pct", "ebitda_margin_pct",
    "effective_tax_rate_pct", "capex_revenue_pct",
]
display(annual[ratio_fields].T.round(2))

debt_fields = [
    "debt_current", "debt_noncurrent", "total_debt", "cash", "net_debt",
    "operating_leases_current", "operating_leases_noncurrent",
    "finance_leases_current", "finance_leases_noncurrent",
]
display(annual[debt_fields].T.round(2))

working_capital = annual[[
    "receivables", "inventory", "accounts_payable",
    "nwc_proxy", "change_nwc_proxy",
]]
display(working_capital.round(2))

## 4. Operating history aligned to the selected years

These observations were migrated from the prior notebook into a source dataset. This removes duplicated arrays and fixes their year alignment; it does not automate retrieval of new production, price, reserve, or WTI observations. A new fiscal year shows missing operating data until a reviewed source record is added. Source and definition notes remain visible below.

In [ ]:
operating, operating_audit = load_operating_observations(
    PROJECT_ROOT / "data/reference/eog_operating_history.json",
    years=annual.index, ticker=TICKER,
)
required_operating = [
    "oil_mbbl_day", "gas_mmcf_day", "ngl_mbbl_day", "total_mboe_day",
    "annual_mmboe", "realized_oil", "realized_gas", "realized_ngl",
    "loe_boe", "gpt_boe", "ga_boe", "taxes_other_than_income",
    "oil_reserves_mmbbl", "ngl_reserves_mmbbl", "gas_reserves_bcf",
    "reserves_mmboe", "wti",
]
operating = operating.reindex(columns=required_operating)
display(operating.T.round(2))
display(operating_audit)

missing_operating = operating.isna().sum(axis=1)
for year, missing in missing_operating.items():
    if missing:
        print(f"FY{year}: {missing} operating fields unavailable; new disclosures needed.")

## 5. Business-driver metrics

All calculations join on fiscal year, with no positional list matching or forward-filling. Production growth for the first displayed year is blank here because this operating table only contains displayed years. Gas is converted at EOG's disclosed 6 Mcf per Boe. EBITDA per Boe uses the selected financial provider and includes that provider's definition limitations.

In [ ]:
drivers = pd.DataFrame(index=annual.index)
total = operating["total_mboe_day"].replace(0, float("nan"))
volume = operating["annual_mmboe"].replace(0, float("nan"))
drivers["Oil mix (%)"] = 100 * operating["oil_mbbl_day"] / total
drivers["Gas mix (%)"] = 100 * (operating["gas_mmcf_day"] / 6) / total
drivers["NGL mix (%)"] = 100 * operating["ngl_mbbl_day"] / total
drivers["Annual production growth (%)"] = 100 * (volume / volume.shift() - 1)
drivers["LOE / Boe (USD)"] = operating["loe_boe"]
drivers["GP&T / Boe (USD)"] = operating["gpt_boe"]
drivers["G&A / Boe (USD)"] = operating["ga_boe"]
drivers["Capex / Boe (USD)"] = annual["capex"] / volume
drivers["Calculated EBITDA / Boe (USD)"] = annual["ebitda"] / volume
drivers["WTI less realized oil (USD/Bbl)"] = operating["wti"] - operating["realized_oil"]
display(drivers.T.round(2))

## 6. Compare the two sources when both caches exist

The comparison uses the same selected years. A missing SEC cache does not prevent exploration of WRDS data. Differences remain labeled for investigation; the notebook does not change the chosen source to make differences disappear. The debt bridge tests whether operating leases explain the observed gap.

In [ ]:
comparison_provider = "sec" if FINANCIAL_SOURCE == "wrds" else "wrds"
comparison_history = None
try:
    comparison_history = load_history(
        TICKER, HISTORY_YEARS, provider=comparison_provider, cik=SEC_CIK,
        end_year=int(annual.index.max()), refresh=False, root=PROJECT_ROOT,
    )
except (HistoryError, SECError) as error:
    print(f"Comparison unavailable: {error}")

if comparison_history is not None:
    sec_annual = annual if FINANCIAL_SOURCE == "sec" else comparison_history.annual
    wrds_annual = annual if FINANCIAL_SOURCE == "wrds" else comparison_history.annual
    comparison_rows = []
    for metric in ["revenue", "ebit", "dd_a", "capex", "operating_cash_flow"]:
        for year in annual.index:
            vendor = wrds_annual.loc[year, metric]
            filed = sec_annual.loc[year, metric]
            comparison_rows.append({
                "Year": year, "Metric": metric, "WRDS": vendor, "SEC": filed,
                "WRDS less SEC": vendor - filed,
            })
    display(pd.DataFrame(comparison_rows).round(2))
    debt_bridge = pd.DataFrame({
        "WRDS debt": wrds_annual["total_debt"],
        "SEC debt": sec_annual["total_debt"],
        "SEC operating leases": sec_annual["operating_leases"],
    })
    debt_bridge["Unexplained difference"] = (
        debt_bridge["WRDS debt"] - debt_bridge["SEC debt"]
        - debt_bridge["SEC operating leases"]
    )
    display(debt_bridge.round(2))

## 6a. Receivables classification and acquisition review

The SEC download uses EOG's reported receivables separately from income taxes receivable. Compustat reports total receivables. A difference matching the tax receivable is evidence of a classification difference; it does not justify applying a blanket adjustment to every year.

The debt bridge above compares Compustat debt with SEC debt plus operating leases. Finance leases are already included in EOG's reported debt. Keep that classification consistent when selecting a valuation debt balance.

Cash paid for business acquisitions is displayed separately from capital expenditures. Year-end NWC movements in acquisition years can include balances acquired with the business. The NWC proxy therefore remains under review before UFCF; no acquisition-related working-capital adjustment is guessed here.


In [ ]:
if comparison_history is not None:
    if "income_taxes_receivable" not in sec_annual.columns:
        raise RuntimeError(
            "The SEC table predates the tax-receivable field. "
            "Run the setup cell and every cell above this one again, "
            "or choose Restart Kernel and Run All."
        )
    receivables_review = pd.DataFrame({
        "WRDS total receivables": wrds_annual["receivables"],
        "SEC reported receivables": sec_annual["receivables"],
        "SEC income taxes receivable": sec_annual["income_taxes_receivable"],
    })
    receivables_review["WRDS less SEC receivables"] = (
        receivables_review["WRDS total receivables"]
        - receivables_review["SEC reported receivables"]
    )
    receivables_review["Difference less reported tax receivable"] = (
        receivables_review["WRDS less SEC receivables"]
        - receivables_review["SEC income taxes receivable"]
    )
    display(receivables_review.round(2))
    print("A zero residual supports the tax-receivable explanation for that year; missing tax data stay blank.")

display(annual[["capex", "acquisitions", "nwc_proxy", "change_nwc_proxy"]].round(2))
acquisition_years = annual.index[annual["acquisitions"].notna() & annual["acquisitions"].ne(0)].tolist()
print("Years with reported acquisition cash flows requiring NWC review:", acquisition_years)


## 7. Historical operating review

Compare the latest selected year with the immediately preceding year. Every value comes from the normalized financial history or the year-indexed operating observations. Absolute changes retain the row's units; changes in margins and production mix are shown in percentage points.

The selected unit-cost subtotal is lease and well + gathering, processing and transportation + G&A. It excludes DD&A, exploration, dry-hole costs, impairments, marketing costs, and taxes other than income. It is not total operating cost. Historical growth includes acquired operations where applicable and is not automatically a forecast assumption.

Use this review to identify which inputs require a separate forecast decision: organic production by commodity, realized prices and differentials, recurring unit costs, capital requirements, and operating working capital. Operating observation retrieval is still a separate step; a missing new year remains blank.


In [ ]:
latest_year = int(annual.index.max())
previous_year = latest_year - 1

if previous_year in annual.index:
    for metric, label in [
        ("revenue", "Revenue"), ("operating_cash_flow", "Operating cash flow"),
        ("capex", "Capital expenditures"),
    ]:
        latest, previous = annual.loc[latest_year, metric], annual.loc[previous_year, metric]
        if pd.notna(latest) and pd.notna(previous) and previous != 0:
            print(f"{label}: {100 * (latest / previous - 1):+.1f}% in FY{latest_year} versus FY{previous_year}.")
        else:
            print(f"{label}: insufficient data for a year-over-year comparison.")
    print(f"Provider used: {history.provider}; inspect reconciliation before mixing sources.")
else:
    print("Select at least two years to compare trends.")


from app.history_review import operating_driver_review

operating_review = operating_driver_review(annual, operating)
display(operating_review.round(2))

for metric in ["Annual production", "Realized oil price", "Realized gas price",
               "Selected unit-cost subtotal", "Capex per Boe", "Calculated EBITDA per Boe"]:
    change = operating_review.loc[metric, "Change (%)"]
    if pd.notna(change):
        print(f"{metric}: {change:+.1f}% versus the preceding year.")
    else:
        print(f"{metric}: comparison unavailable; inspect source coverage.")

margin_change = operating_review.loc["Operating margin", "Change"]
if pd.notna(margin_change):
    print(f"Operating margin changed {margin_change:+.2f} percentage points.")

acquisition_cash = annual.loc[latest_year, "acquisitions"]
if pd.notna(acquisition_cash) and acquisition_cash != 0:
    print(f"FY{latest_year} includes reported acquisition cash flows of ${acquisition_cash:,.0f} million.")
    print("Separate acquired production and acquired working capital before treating the changes as recurring.")
elif pd.isna(acquisition_cash):
    print("Acquisition cash-flow data are missing; organic growth has not been established.")
else:
    print("No acquisition cash outflow is reported in this row; that alone does not establish organic growth.")

print("Capex per Boe uses total annual production. A lower ratio alone does not establish lower development costs.")
print("Next: quantify acquisition effects and finalize operating NWC before setting forecast assumptions.")


## 8. Acquisition effects: establish the historical base for forecasting

An acquisition changes production, expenses and balance-sheet accounts on different timelines. We first explain those effects before choosing recurring FCF assumptions.

This section reads **the latest selected year's annual filing**, resolved from SEC submissions metadata. Its inline XBRL includes acquisition dimensions that the companyfacts API omits. The company adapter maps tags and transaction contexts; it does not contain financial amounts, acquisition dates or fixed year ranges.

The document is cached separately from companyfacts. `REFRESH_FILING = False` uses the saved document. For a newly selected filing, set it to `True` once, then back to `False`. Refresh SEC metadata in the setup cell first to discover newly filed reports. Unsupported or ambiguous disclosures need review; a missing deal is not evidence that no acquisition occurred.


In [ ]:
import app.sec_filings as filings_module
import app.acquisition_review as acquisition_module
importlib.reload(filings_module)
importlib.reload(acquisition_module)

REFRESH_FILING = False
acquisition_tables = {}
acquisition_audit = pd.DataFrame()

try:
    if TICKER.upper() != "EOG" or history.provider.upper() != "SEC":
        raise ValueError("This filing adapter currently requires EOG and FINANCIAL_SOURCE = 'sec'.")
    filing_bytes, filing_metadata, filing_path = filings_module.load_annual_filing(
        SEC_CIK, TICKER, int(annual.index.max()),
        refresh=REFRESH_FILING, root=PROJECT_ROOT,
    )
    acquisition_deals, acquisition_facts, acquisition_audit = acquisition_module.read_eog_acquisitions(
        filing_bytes, filing_metadata,
    )
    acquisition_tables = acquisition_module.acquisition_bridges(
        annual, operating, acquisition_deals, acquisition_facts, provider=history.provider,
    )
    print(f"Fiscal period end: {filing_metadata['report_date']} | Filed: {filing_metadata['filed']}")
    print(f"Retrieved UTC: {filing_metadata['retrieved_at_utc']}")
    display(Markdown(f"Source: [EOG annual filing]({filing_metadata['url']})"))
    print(f"Saved document: {filing_path.relative_to(PROJECT_ROOT)}")
    if acquisition_tables:
        display(acquisition_tables["deals"].round(2))
    else:
        print("No supported current-year acquisition was mapped. Review the filing before concluding there were none.")
except (SECError, ValueError, OSError) as error:
    print(f"Acquisition review unavailable: {error}")


### 8a. Acquisition cash and working-capital bridge

The bridge first compares mapped consideration **less acquired cash** with reported acquisition cash outflows. If they do not reconcile, the residual NWC calculation stays unavailable.

For the same narrow definition used above:

- Acquired NWC proxy = acquired receivables + inventory - acquired payables.
- Remaining NWC change = reported annual change - acquired NWC proxy.

A negative remaining change indicates a release in this balance-sheet proxy. It is **not yet the DCF cash-flow input**: capital payables, noncash movements, joint-interest balances and omitted operating accounts remain to be reviewed. A cash tie supports coverage of mapped cash acquisitions; it does not establish that all noncash transactions or other balance changes have been captured.

These are values as disclosed in the selected 10-K. Purchase-price allocations can be preliminary; subsequent quarterly revisions are not retrieved by this section.


In [ ]:
if acquisition_tables:
    display(acquisition_tables["nwc"].round(2))
else:
    print("Load supported acquisition disclosures in section 8 before using this bridge.")


### 8b. Review transaction expenses without changing the historical statements

The candidate column removes only acquisition costs explicitly tagged to **G&A**. Interest transaction costs are shown separately in the deal table and are not added to EBIT or EBITDA. The broader discussion of professional services and other expense changes is not treated as an acquisition-cost addback.

This is an illustrative expense adjustment, not an approved recurring cost assumption or a full-year combined-company forecast. Annual production still includes acquired output only after closing. Historical `annual`, `operating` and `drivers` values remain the reported series.


In [ ]:
if acquisition_tables:
    display(acquisition_tables["expense_candidates"].round(2))
    print("Candidate only: decide whether the disclosed expense is appropriate to exclude before forecasting.")
else:
    print("Expense candidates are unavailable until the acquisition disclosures are loaded.")


### 8c. Distinguish post-closing results from full-year pro forma information

The company's **unaudited pro forma** revenue and net income describe a hypothetical earlier closing. They are not observed full-year acquired production, EBITDA or free cash flow. Do not annualize the post-closing contribution by multiplying by a fraction of the year.

In the audit table, an acquiree's since-closing contribution can have a full-year XBRL context. Its economic coverage still begins on the acquisition date shown in the deal table. The metric name and source note govern that distinction.


In [ ]:
if acquisition_tables:
    display(acquisition_tables["proforma"].round(2))
    display(Markdown("**Extraction audit: source tags, transaction dimensions, dates and unit scales**"))
    display(acquisition_audit)
else:
    print("No supported acquisition contribution or pro forma disclosures are available.")


### What this means for the next step

We have a sourced acquisition bridge and a visible expense adjustment candidate. We still need a forecast base that reflects the assets owned for the forecast period.

1. Build the oil, gas and NGL volume/realized-price revenue bridge. Separate upstream sales from marketing activity and derivatives.
2. Refresh production disclosures and current guidance, including acquired operations and subsequent asset sales. Do not label reported production growth as organic without supporting data.
3. Review capital payables and cash-flow working-capital disclosures, plus recurring unit costs, DD&A and capital needs. Capex per Boe alone does not establish sustaining capex or development economics.
4. Then select explicit production, commodity price, cost, tax, capex and NWC assumptions to calculate forecast UFCF. WACC and valuation follow those operating forecasts.

Gas-to-Boe conversion compares energy content, not dollar value. The forecast will model the three commodities separately.


## 9. Oil, natural gas and NGL revenue model: historical reconstruction

We now connect operating quantities to revenue, using the same economic relationships needed for the forecast. **These are historical calculations; no future growth or commodity prices have been assumed.**

This section retrieves daily production, composite realized prices, reported commodity sales and other revenue components directly from annual filing documents. It uses each report's comparative years to cover the selected history window. The observations are new, independently sourced inputs called `revenue_disclosures`; they do not overwrite the earlier operating reference dataset.

All financial amounts below are **USD millions**, unless another unit is shown. Crude oil includes condensate. Prices exclude the financial derivative effects reported separately in the income statement.

`REFRESH_REVENUE_FILINGS = False` uses saved documents. To retrieve documents for a newly selected window, refresh SEC metadata in the setup cell first, set this flag to `True` once, then return it to `False`. No financial values or fixed year lists are embedded in the code.


In [ ]:
import app.eog_revenue as revenue_source_module
import app.energy_revenue as revenue_model_module
importlib.reload(revenue_source_module)
importlib.reload(revenue_model_module)

REFRESH_REVENUE_FILINGS = False
revenue_disclosures = None
revenue_precision = None
revenue_audit = pd.DataFrame()
commodity_revenue = None
revenue_bridge = None

try:
    if history.provider.upper() != "SEC":
        raise ValueError("Use FINANCIAL_SOURCE = 'sec' for this filing revenue bridge.")
    revenue_disclosures, revenue_precision, revenue_audit = revenue_source_module.load_revenue_history(
        annual.index, cik=SEC_CIK, ticker=TICKER,
        refresh=REFRESH_REVENUE_FILINGS, root=PROJECT_ROOT,
    )
    commodity_revenue, revenue_bridge = revenue_model_module.build_revenue_bridge(
        revenue_disclosures, revenue_precision, annual, provider=history.provider,
    )
    display(Markdown("**Production and realized prices extracted from the filings**"))
    display(revenue_disclosures[revenue_source_module.OPERATING_FIELDS].T.round(2))
    display(Markdown("**Source coverage by fiscal year**"))
    display(revenue_audit[["year", "filing_report_date", "accession", "filed", "retrieved_at_utc"]].drop_duplicates().sort_values("year"))
except (SECError, ValueError, OSError) as error:
    print(f"Revenue model unavailable: {error}")


### 9a. Calculate sales from production and realized prices

| Commodity | Annual volume calculation | Revenue in USD millions |
|---|---|---|
| Oil and condensate | MBbl/day x days / 1,000 = MMBbl | MMBbl x USD/Bbl |
| Natural gas | MMcf/day x days / 1,000 = Bcf | Bcf x USD/Mcf |
| NGLs | MBbl/day x days / 1,000 = MMBbl | MMBbl x USD/Bbl |

The gas formula works because one Bcf contains one million Mcf. **No Boe conversion is used to price gas.** Days are calculated from each fiscal year, including leap years. Acquired production is already included in the reported daily average only for the period owned; do not add it a second time or annualize it here.

Published daily volumes and prices are rounded. `Calculated less reported` retains the resulting difference. The displayed rounding bound assumes rounding to the nearest published unit for volume, price and reported sales:

`days / 1,000 x (abs(volume) x half-price-unit + abs(price) x half-volume-unit + half-volume-unit x half-price-unit) + half-sales-unit`

A difference within this bound is consistent with published rounding; it does not prove rounding is the only cause. Missing inputs stay unavailable, and larger differences remain flagged for review.


In [ ]:
if commodity_revenue is not None:
    display(commodity_revenue.round(3))
    print(commodity_revenue["Status"].value_counts().to_string())
    print("Calculated sales, reported sales, differences and rounding bounds are USD millions.")
else:
    print("Load the section 9 revenue disclosures first.")


### 9b. Reconcile upstream sales to total company revenue

The first reconciliation uses **reported oil + gas + NGL sales**. Add gathering/processing/marketing revenue, signed derivative gains or losses, signed asset-disposition gains or losses, and other revenue to obtain the filed total. The two residual rows show whether that total matches both the component sum and our separate SEC financial-history table.

Marketing revenue includes third-party commodity sales, fees and sand activity. Marketing costs are shown separately below the revenue bridge. **Marketing revenue less marketing costs is only a subtotal**, not standalone segment EBITDA; other costs can be classified elsewhere. If we later model marketing on a net basis, we must also remove the corresponding marketing costs from the expense build to avoid double counting.

Financial derivative gains/losses include cash settlements and valuation changes. The signed income-statement amount is used only to reconcile historical revenue; it is not automatically a recurring revenue or cash-flow forecast. Asset-sale gains also differ from cash sale proceeds.


In [ ]:
if revenue_bridge is not None:
    display(revenue_bridge.T.round(3))
    revenue_reconciliation = revenue_bridge[[
        "Component sum less filing total", "Filing total less SEC annual history",
    ]].copy()
    revenue_reconciliation["Status"] = revenue_reconciliation.apply(
        lambda row: "Missing input" if row.isna().any() else
        "Ties" if row.abs().le(0.05).all() else "Review difference", axis=1,
    )
    display(revenue_reconciliation)
    print("Each residual is USD millions; reconciliation tolerance is $0.05 million.")
else:
    print("Revenue reconciliation is unavailable until section 9 loads successfully.")


### 9c. Trace the inputs and prepare the forecast decisions

The audit records filing, reporting period, original text, scale, units, published precision, and the XBRL product dimension or labeled table row. All fields for a year come from one selected filing. Earlier comparative years may have later restatements; the separate SEC annual-total comparison makes total-revenue differences visible.

The supported filing layouts now automate retrieval of these six production/price inputs and the revenue components. Reserves, unit costs and the other earlier operating observations still require their own refresh workflow. An unfamiliar layout or missing tag requires review rather than a substituted value.

**Next:** establish forward oil, gas and NGL volumes using current production/guidance and the assets owned for the forecast period, then choose realized-price assumptions and differentials. Encino acquisition timing and subsequent asset sales still matter. The historical growth rates and rounded-price reconciliation differences are not forecast assumptions.


In [ ]:
if not revenue_audit.empty:
    display(revenue_audit.sort_values(["year", "metric"]))
else:
    print("No revenue source audit is available.")


## 10. Operating costs, DD&A, EBITDA and EBIT

The revenue bridge gives us the top of the historical operating model. Next we explain the expenses that turn that revenue into operating profit. This section pulls the expense lines directly from the same annual filing selection used for revenue, with no hardcoded financial amounts or fixed year range.

We already retrieved total DD&A and operating income in the financial statements. Here we rebuild operating income from its components and show how DD&A fits into that calculation. **These remain historical results, not approved recurring forecast assumptions.**

`REFRESH_COST_FILINGS = False` uses the saved documents. For a newly selected window, refresh SEC metadata first, set this flag to `True` once, then return it to `False`.


In [ ]:
import app.eog_revenue as revenue_source_module
import app.eog_costs as cost_source_module
import app.energy_costs as cost_model_module
importlib.reload(revenue_source_module)
importlib.reload(cost_source_module)
importlib.reload(cost_model_module)

REFRESH_COST_FILINGS = False
cost_disclosures = None
cost_audit = pd.DataFrame()
cost_bridge = None
cost_unit_metrics = None
cost_reconciliation = None

try:
    if history.provider.upper() != "SEC":
        raise ValueError("Use FINANCIAL_SOURCE = 'sec' for the operating-cost bridge.")
    cost_disclosures, cost_audit = cost_source_module.load_cost_history(
        annual.index, cik=SEC_CIK, ticker=TICKER,
        refresh=REFRESH_COST_FILINGS, root=PROJECT_ROOT,
    )
    cost_bridge, cost_unit_metrics, cost_reconciliation = cost_model_module.build_cost_bridge(
        cost_disclosures, annual, provider=history.provider,
    )
    display(Markdown("**Operating-cost build, USD millions**"))
    display(cost_bridge.T.round(2))
except (SECError, ValueError, OSError) as error:
    print(f"Operating-cost review unavailable: {error}")


### 10a. How the profit bridge works

1. Sum lease/well, gathering-processing-transportation, exploration, dry holes, impairments, marketing, G&A, taxes other than income, and DD&A.
2. **Revenue - all those operating expenses = operating income**, our operating-income definition of EBIT.
3. **Operating income + DD&A = calculated EBITDA on an operating-income basis.**

Interest expense, nonoperating interest income and income taxes are outside this bridge. The result can differ from a company's income-before-interest-and-taxes subtotal that includes nonoperating income, or from its separately defined adjusted EBITDA.

**DD&A** means depreciation, depletion and amortization. EOG's oil-and-gas property DD&A uses production and reserve bases; acquisitions and changes in reserve estimates can affect the rate. DD&A is an accounting expense for the consumption of capitalized assets. It is added back when moving toward cash flow, but capital spending still has to be deducted separately. DD&A is not a substitute for maintenance or development capex.

The subtotal excluding DD&A still contains impairments and potentially other noncash expenses. It is not labeled cash operating cost. We retain those expenses and the revenue-side derivative/disposition gains and losses until specific forecast adjustments are reviewed. The acquisition G&A candidate in section 8 has not been applied here.

Source: the selected filing's income statement and accounting policy notes; individual fact links are in the audit below.


### 10b. Turn historical expenses into visible operating drivers

The calculated production denominator is:

`(oil MBbl/day + NGL MBbl/day + gas MMcf/day / 6) x days in year / 1,000 = MMBoe`

USD millions divided by MMBoe gives USD/Boe. This denominator comes from rounded daily volumes and can differ slightly from the filing's separately rounded annual total. The gas-to-Boe conversion is for cost intensity; section 9 continues to price gas separately in USD/Mcf.

- Lease/well and GP&T per Boe help describe production and delivery costs.
- G&A per Boe is a scale diagnostic; G&A is not assumed to vary proportionally with production.
- DD&A per Boe is a historical reference for a later depreciation forecast.
- Exploration, dry holes and impairment per Boe show historical burden, not automatic recurring unit-cost assumptions.
- Taxes other than income divided by upstream sales is a diagnostic ratio. That tax line includes property, payroll and other taxes as well as production taxes; it is not a pure severance-tax rate.

Marketing costs are excluded from per-Boe drivers because they include purchases of third-party commodities. They remain included once in the full operating expense bridge.


In [ ]:
if cost_unit_metrics is not None:
    display(cost_unit_metrics.T.round(3))
else:
    print("Load the operating-cost disclosures in section 10 first.")


### 10c. Reconcile expense totals and profit measures

Each year checks the sum of detailed expenses, rebuilt EBIT and calculated EBITDA against filing totals. The filing's revenue, expenses, operating income and DD&A also compare to the separate SEC annual history. Missing inputs never pass. Differences remain visible for investigation.

Two company-specific mapping details matter:

- Older filings report gathering/processing and transportation separately. We combine both only when the combined GP&T line is absent; missing components do not become zero.
- EOG uses the XBRL tag `OperatingLeaseExpense` for its **Lease and Well** statement line. This mapping follows that filing label; it is not the separate lease-accounting footnote's total lease cost.


In [ ]:
if cost_reconciliation is not None:
    display(cost_reconciliation)
    print(cost_reconciliation["Status"].value_counts().to_string())
    print("Residuals and the $0.05 reconciliation tolerance are in USD millions.")
    display(Markdown("**GP&T source definitions across the selected history**"))
    display(cost_audit.loc[cost_audit["metric"].isin(["gpt", "transportation", "gathering_processing"]),
                          ["year", "metric", "value", "units", "source_field", "source"]].sort_values(["year", "metric"]))
else:
    print("Operating-cost reconciliation is unavailable.")


### 10d. Source audit and next step toward free cash flow

The audit below contains expense inputs, source totals and the production/revenue context used in the calculations. The original financial history remains unchanged.

**Completed here:** the historical revenue-to-operating-profit bridge, including DD&A and a clearly defined EBITDA calculation.

**Next in the cash-flow build:** review operating taxes, capital expenditures and operating working capital. We must resolve the remaining acquisition/capital-payable issues and distinguish capital spending from business purchases before treating an NWC change or reinvestment amount as a forecast FCF input. Then we can set forward production, prices, recurring costs, DD&A, tax and reinvestment assumptions together.

The eventual formula is `EBIT x (1 - operating tax rate) + DD&A - capex - change in operating NWC`, with explicit treatment of any other required noncash adjustments. We are not calculating a valuation from these historical profit subtotals yet.


In [ ]:
if not cost_audit.empty:
    display(cost_audit.sort_values(["year", "metric"]))
else:
    print("No cost source audit is available.")


## 11. Historical cash flow: taxes, reinvestment and working capital

We now trace net income into operating cash flow and explain investing outflows. The inputs are extracted from the annual filings selected for the revenue/cost history, including company-specific cash-flow tags. Amounts are **USD millions**, except percentages.

This section separates reported tax expense from cash taxes, property additions from business acquisitions, and balance-sheet movements from cash-flow contributions. It does not choose a forecast tax rate or assume that a working-capital proxy is ready for the DCF.

`REFRESH_CASHFLOW_FILINGS = False` uses the saved documents. Refresh SEC metadata first before setting it to `True` once for a new window. Missing disclosures stay unavailable, including acquisition fields absent in older selected filings.


In [ ]:
import app.eog_revenue as revenue_source_module
import app.eog_cashflow as cashflow_source_module
import app.energy_cashflow as cashflow_model_module
importlib.reload(revenue_source_module)
importlib.reload(cashflow_source_module)
importlib.reload(cashflow_model_module)

REFRESH_CASHFLOW_FILINGS = False
cashflow_disclosures = None
cashflow_audit = pd.DataFrame()
cashflow_review = {}

try:
    if history.provider.upper() != "SEC":
        raise ValueError("Use FINANCIAL_SOURCE = 'sec' for the cash-flow review.")
    cashflow_disclosures, cashflow_audit = cashflow_source_module.load_cashflow_history(
        annual.index, cik=SEC_CIK, ticker=TICKER,
        refresh=REFRESH_CASHFLOW_FILINGS, root=PROJECT_ROOT,
    )
    cashflow_review = cashflow_model_module.build_cashflow_review(
        cashflow_disclosures, annual, provider=history.provider,
    )
    display(cashflow_audit[["year", "filing_report_date", "accession", "filed", "retrieved_at_utc"]].drop_duplicates().sort_values("year"))
except (SECError, ValueError, OSError) as error:
    print(f"Cash-flow review unavailable: {error}")


### 11a. Tax expense is not cash tax paid

The tax table reconciles **current + deferred + other tax provision** to reported income tax expense. The separate cash-tax disclosure reflects payment timing, refunds and the scope described in the filing.

Tax-credit purchases are displayed as a memo. In the reviewed filing, disclosed cash taxes already include payments to buy those credits, so we do not add the memo again. A blank memo in an earlier year means the supported tag was unavailable, not that purchases were zero.

The effective book rate and cash-taxes-to-pretax-income ratio describe history. Neither is automatically the forward operating tax rate. Ratios stay unavailable when pretax income is zero or negative. Tax on unlevered operating income will require a separate assumption that addresses financing effects and timing.


In [ ]:
if cashflow_review:
    display(cashflow_review["taxes"].T.round(2))
else:
    print("Load section 11 before reviewing taxes.")


### 11b. Rebuild operating cash flow from net income

The bridge includes DD&A, impairments, stock compensation, deferred taxes, dry-hole costs, reversals of accounting gains/losses, actual derivative settlements, and changes in operating accounts. The final residual compares the rebuilt amount with reported CFO.

**Signs matter:** positive cash contributions increase CFO. An increase in an operating asset generally consumes cash, while an increase in a payable generally provides cash. XBRL tag signs are preserved in the audit, then converted explicitly by the sign guide. Accounting derivative gains/losses are reversed before actual settlements are included.

These historical reconciliation adjustments are not automatic valuation addbacks. Stock compensation still has an economic cost; dry-hole costs must be considered consistently with exploration spending; impairments and derivative valuation changes require separate forecast treatment. The bridge starts from net income to reproduce reported CFO, which retains financing and tax effects.


In [ ]:
if cashflow_review:
    display(cashflow_review["cfo"].T.round(2))
    display(Markdown("**How source values become signed cash-flow contributions**"))
    display(cashflow_review["sign_guide"])
else:
    print("The operating cash-flow bridge is unavailable.")


### 11c. Property additions, investing working capital and acquisitions

The earlier `annual["capex"]` field sums oil/gas and other PP&E addition lines. EOG separately reports an investing-related working-capital contribution. We therefore show both:

- Property additions before investing WC.
- Property additions **less the signed investing-WC inflow**. A negative contribution increases the resulting outflow.

The complete investing reconciliation also includes business acquisitions, asset-sale proceeds and any separately disclosed other investing activity. It sums the actual lines present in each selected cash-flow statement. An absent category remains blank; it is not silently populated with zero. An unfamiliar new investing line stops the reconciliation for review.

The investing-WC entry in CFO and its opposite entry in investing cash flow are shown separately and checked to offset. Their treatment must remain consistent when defining forecast capex and operating NWC; do not apply the same capital-payable movement twice.

**CFO less property spending is a historical cash-availability diagnostic, not unlevered FCF or EOG's separately defined non-GAAP FCF.** The displayed subtotal excludes business acquisition payments and asset-sale proceeds. It still reflects cash interest, historical tax timing and other operating cash-flow items.

Capitalized interest is shown as a memo and is not added to spending again. Its financing treatment, growth versus maintenance spending, asset-retirement cash costs and dry-hole classification remain explicit decisions for the forecast.


In [ ]:
if cashflow_review:
    display(cashflow_review["investing"].T.round(2))
else:
    print("The capital-spending and investing bridge is unavailable.")


### 11d. Compare working-capital definitions before selecting one

The balance-sheet proxy is receivables + inventory - payables, and its change is positive for investment. The cash-flow table instead uses **positive for cash inflow**. We show the negative of the narrow three-account cash contribution to make its investment-positive convention explicit.

The six-account cash contribution also includes accrued taxes and other assets/liabilities. It is broader than operating NWC suitable for a DCF: subtracting cash taxes separately and then including tax-payable changes again could double count tax timing. Other accounts can contain non-operating or noncurrent items.

The acquisition-adjusted balance-sheet diagnostic from section 8 and the cash-flow measures answer different questions. They should not be forced to match with a plug. We retain all definitions, with investing-WC reclassification shown separately; none is automatically selected as forecast change in NWC.


In [ ]:
if cashflow_review:
    display(cashflow_review["working_capital"].T.round(2))
    if globals().get("acquisition_tables"):
        display(Markdown("**Earlier acquisition/NWC diagnostic, retained for comparison**"))
        display(acquisition_tables["nwc"].round(2))
else:
    print("The working-capital comparison is unavailable.")


### 11e. Reconciliation, source audit and the forecast handoff

Checks cover CFO, investing cash flow, tax expense, opposite investing-WC entries, and agreement with the separate SEC annual history. Missing comparisons are reported as missing rather than passed. Every amount remains traceable to its filing/tag; the investing rebuild records the actual component formula.

**Section 12 builds the forward assumptions table.** Historical revenue, expense and cash-flow relationships are now visible. Before calculating forecast UFCF, that table must explicitly address:

| Forecast input | Evidence now available | Decision still needed |
|---|---|---|
| Oil, gas and NGL production | Historical volumes and acquisition timing | Current guidance, assets owned, decline and development assumptions |
| Realized prices | Historical prices and sales reconstruction | Commodity-price paths and differentials |
| Operating costs and DD&A | Detailed expenses and unit-cost diagnostics | Recurring cost basis, impairments and production/depletion relationship |
| Operating taxes | Current/deferred provisions and cash-tax history | Unlevered operating tax assumption and timing |
| Reinvestment | Property additions, investing WC and acquisition cash | Maintenance/growth needs, capitalized interest and consistent cash/accrual basis |
| Operating NWC | Balance-sheet, acquisition and cash-flow diagnostics | Accounts included, tax exclusion and capital-payable treatment |

That forecast will use `NOPAT + DD&A - capex - change in operating NWC`, with any additional required noncash adjustments shown explicitly. No historical cash subtotal in this section is substituted for forecast UFCF.


In [ ]:
if cashflow_review:
    display(cashflow_review["checks"])
    print(cashflow_review["checks"]["Status"].value_counts().to_string())
    print("Residuals and the $0.05 reconciliation tolerance are in USD millions.")
    display(cashflow_audit.sort_values(["year", "metric"]))
else:
    print("No cash-flow reconciliation or source audit is available.")


## 12. Forward assumptions table

Sections 1-11 explained what **happened**. From here on we build what we **expect**, so every number must say where it came from. Five kinds of input appear, and the table labels each one:

| Status | Meaning | Example |
|---|---|---|
| Reported / Sourced fact | Filed actual results | Nine months of 2026 from the Q3 earnings release |
| Management guidance | EOG's own forecast in an SEC filing | Q4 2026 production and unit-cost ranges |
| Benchmark forecast | A third-party price outlook | EIA Short-Term Energy Outlook (STEO) |
| Historical relationship | A median of our Sections 9-11 history | Net marketing margin, NWC as % of sales |
| Analyst assumption | **Your** choice, with a written reason | Volume growth after 2026, long-term oil price, terminal growth |

**Dates.** The valuation date and horizon are in `data/assumptions/EOG_assumptions.json`. FY2026 is **not** an actual year: it is nine reported months plus a fourth-quarter forecast. The explicit forecast covers FY2027-FY2031 (five years, per the master spec). Only the part of Q4 2026 after the valuation date is discounted.

**Where guidance comes from.** EOG printed its Q4/FY2026 guidance table as an image inside the 8-K. Those values were transcribed into `data/reference/eog_guidance.json`; the code confirms the cached image has the same SHA-256 fingerprint as the reviewed page, but it cannot re-read the numbers, so the audit says "manually transcribed". Sentences quoted from the 10-Q are checked word-for-word against the cached filing. EOG's "low single-digit 2027 oil growth" comment came from the earnings call, not an SEC filing, so 2027+ volumes are analyst assumptions.

**To change an assumption**, edit `data/assumptions/EOG_assumptions.json` (a number, or a rule such as the guidance midpoint), save it, and rerun Sections 12-16. A driver marked `"status": "Decision needed"` stops the forecast until you choose a value.

Refresh flags work like earlier sections: `False` uses saved files; set one to `True` once to download a new snapshot, then set it back.

In [ ]:
import json
import app.sec_filings as filings_module
import app.eog_quarterly as quarterly_module
import app.guidance as guidance_module
import app.eia as eia_module
import app.market_inputs as market_module
import app.forecast_assumptions as assumptions_module
import app.energy_forecast as forecast_module
import app.valuation as valuation_module
import app.dcf_workflow as workflow_module
for module in (filings_module, quarterly_module, guidance_module, eia_module, market_module,
               assumptions_module, forecast_module, valuation_module, workflow_module):
    importlib.reload(module)
from datetime import date

REFRESH_GUIDANCE = False   # True once: download the guidance exhibits/images and 10-Q into the SEC cache
REFRESH_EIA = False        # True once: download the latest EIA STEO workbook
REFRESH_MARKET = False     # True once: new FMP price, Treasury yield and Damodaran snapshots

GUIDANCE_PATH = PROJECT_ROOT / "data/reference/eog_guidance.json"
ASSUMPTIONS_PATH = PROJECT_ROOT / f"data/assumptions/{TICKER}_assumptions.json"

forecast_inputs_ready = False
try:
    if history.provider.upper() != "SEC" or TICKER.upper() != "EOG":
        raise ValueError("Sections 12-16 use EOG's SEC filings; set TICKER = 'EOG' and FINANCIAL_SOURCE = 'sec'.")
    selections = assumptions_module.load_selections(ASSUMPTIONS_PATH)
    VALUATION_DATE = date.fromisoformat(selections["valuation"]["valuation_date"])
    BASE_YEAR = int(selections["valuation"]["base_year"])
    STUB_QUARTER = int(selections["valuation"]["stub_quarter"])
    FORECAST_YEARS = assumptions_module.forecast_years(selections)

    guidance = guidance_module.load_guidance(GUIDANCE_PATH, cik=SEC_CIK, ticker=TICKER,
                                             refresh=REFRESH_GUIDANCE, root=PROJECT_ROOT)
    guidance_sources = json.loads(GUIDANCE_PATH.read_text(encoding="utf-8"))["sources"]
    supplemental = guidance_sources["q3_2026_supplemental"]
    quarterly, quarter_balances, quarterly_audit, quarterly_checks = quarterly_module.load_supplemental(
        SEC_CIK, TICKER, supplemental["accession"], supplemental["document"],
        refresh=REFRESH_GUIDANCE, root=PROJECT_ROOT)
    ytd_label = f"{BASE_YEAR}YTD"
    if ytd_label not in quarterly.index or f"{BASE_YEAR}Q{STUB_QUARTER - 1}" not in quarterly.index:
        raise ValueError(f"The supplemental tables do not end with {BASE_YEAR} Q{STUB_QUARTER - 1}; update the assumptions dates.")
    nine_month = quarterly.loc[ytd_label]
    latest_balance = quarter_balances.iloc[-1].copy()
    latest_balance.name = quarter_balances.index[-1]
    if latest_balance.name != selections["valuation"]["balance_sheet_date"]:
        raise ValueError("Latest supplemental balance sheet differs from balance_sheet_date in the assumptions file.")

    steo_bytes, steo_metadata = eia_module.load_steo(refresh=REFRESH_EIA, root=PROJECT_ROOT)
    steo_monthly, steo_info = eia_module.parse_steo(steo_bytes)
    steo_annual = eia_module.annual_prices(steo_monthly)
    steo_stub = eia_module.quarter_prices(steo_monthly, BASE_YEAR, STUB_QUARTER)

    treasury = market_module.treasury_10y(VALUATION_DATE, refresh=REFRESH_MARKET, root=PROJECT_ROOT)
    erp_reference = market_module.implied_erp(refresh=REFRESH_MARKET, root=PROJECT_ROOT)
    beta_reference = market_module.industry_beta(refresh=REFRESH_MARKET, root=PROJECT_ROOT)
    price_snapshot = market_module.fmp_profile(TICKER, refresh=REFRESH_MARKET, root=PROJECT_ROOT)
    cover = market_module.cover_shares(SEC_CIK, TICKER, root=PROJECT_ROOT)
    aro_settled = {}
    for year in annual.index:
        fact = market_module.xbrl_duration(SEC_CIK, TICKER, "AssetRetirementObligationLiabilitiesSettled",
                                           f"{year}-01-01", f"{year}-12-31", root=PROJECT_ROOT)
        aro_settled[year] = fact["value"] if fact else float("nan")
    forecast_inputs_ready = True

    sources = pd.DataFrame([
        {"Input": "Assumptions file", "Source": str(ASSUMPTIONS_PATH.relative_to(PROJECT_ROOT)), "As of": selections.get("reviewed")},
        {"Input": "Guidance records", "Source": f"{len(guidance)} records, 8-K/10-Q filed {guidance.filed.max()}", "As of": guidance.retrieved_at_utc.max()},
        {"Input": "Quarterly actuals", "Source": f"{supplemental['form']} {supplemental['accession']} EX-99.1", "As of": quarterly_audit.retrieved_at_utc.iloc[0]},
        {"Input": "Price deck", "Source": f"EIA STEO {steo_info['release']} (history through {steo_info['last_historical_month']})", "As of": steo_metadata["retrieved_at_utc"]},
        {"Input": "10-year Treasury", "Source": f"{treasury['value']:.2%} on {treasury['observation_date']}", "As of": treasury["retrieved_at_utc"]},
        {"Input": "Share price", "Source": f"${price_snapshot['price']:.2f} ({price_snapshot['source']})", "As of": price_snapshot["retrieved_at_utc"]},
        {"Input": "Industry beta / ERP", "Source": f"Damodaran ({beta_reference['date_of_analysis']}); ERP to end-{erp_reference['latest_year_end']}", "As of": beta_reference["retrieved_at_utc"]},
        {"Input": "Cover-page shares", "Source": f"{cover['shares']:.1f}M as of {cover['as_of']} ({cover['form']} {cover['accession']})", "As of": cover["filed"]},
    ])
    print(f"Valuation date {VALUATION_DATE}; base year FY{BASE_YEAR}E = 9M reported + Q{STUB_QUARTER} forecast; "
          f"explicit years FY{FORECAST_YEARS[0]}-FY{FORECAST_YEARS[-1]}")
    display(sources)
except (SECError, guidance_module.GuidanceError, eia_module.EIAError, market_module.MarketDataError,
        assumptions_module.AssumptionError, ValueError, OSError, KeyError) as error:
    print(f"Forward inputs unavailable: {error}")

### 12a. The starting point: nine reported months and the asset base after deals

The Q3 2026 earnings release (8-K exhibit 99.1) contains quarterly income statement, volume, price, balance sheet and cash-flow tables. We parse them by row label, then check that **Q1 + Q2 + Q3 = year-to-date** for every dollar line (within $1 million rounding per printed figure).

Asset changes since the FY2025 10-K, from the 10-Q text:
- **Barnett Shale sale** closed September 15, 2026; **Woodford Oil Window sale** closed October 19, 2026 (after the balance-sheet date). Together about 16 MBoe/d in Q3, of which about 2 MBbl/d was oil.
- No new acquisitions are disclosed for 2026. Encino (acquired August 1, 2025) is now in the base for all of 2026.

Because Q4 2026 guidance is given **after** these sales, we grow forecast volumes from the **Q4 2026 guided rate**, not from the full-year average. That avoids treating sold production as if EOG still owned it, and it does not assume reported growth was organic.

In [ ]:
if forecast_inputs_ready:
    key_rows = ["oil_mbbl_day", "ngl_mbbl_day", "gas_mmcf_day", "total_mboe_day", "realized_oil", "realized_ngl",
                "realized_gas", "upstream_sales" if "upstream_sales" in quarterly else "oil_sales", "reported_revenue",
                "reported_operating_income", "dd_a", "operating_cash_flow", "capex_nongaap"]
    shown = [c for c in quarterly.index if c.startswith(str(BASE_YEAR - 1) + "Q4") or c.startswith(str(BASE_YEAR))]
    display(Markdown("**Recent quarters and year-to-date, as reported (USD millions; volumes per day; prices per unit)**"))
    display(quarterly.loc[shown, [r for r in key_rows if r in quarterly]].T.round(2))
    display(Markdown("**Quarter-end balances used for the equity bridge and opening NWC (USD millions)**"))
    display(quarter_balances.tail(3).T)
    print(quarterly_checks.Status.value_counts().to_string())
    display(guidance[guidance.id.str.startswith("divest")][["id", "point", "units", "basis", "verification"]])
else:
    print("Load the forward inputs above first.")

### 12b. Management guidance and how it was verified

Each row names its filing. "Image hash verified" means the cached page image is byte-for-byte the one that was reviewed; the numbers were typed from it. "Quoted text found" means the exact sentence appears in the cached 10-Q. Guidance covers **Q4 2026 and FY2026 only**.

In [ ]:
if forecast_inputs_ready:
    display(guidance[["id", "period", "low", "high", "point", "units", "basis", "accession", "verification"]])

### 12c. The forward assumptions table

`history_references` turns the Section 9-11 history into the same units as each driver, so the table can show a historical reference next to every choice. Rules in the assumptions file are then resolved:

- `guidance` = the printed midpoint for Q4 or FY2026,
- `steo_annual` = EIA's annual average (and, after EIA's horizon, the explicit long-term price you set),
- `historical_median` = the median of the five reported years.

Two things to notice:
1. **Prices.** The STEO expects WTI to fall from about $97 in Q4 2026 toward $70 by December 2027. After 2027 the deck is held flat at the long-term price in the assumptions file. This one choice moves value more than anything else (see the WTI sensitivity in Section 16).
2. **Capex scales with production** at FY2026's guided capex per Boe, inflated 2% a year. That is a simple proxy, not proof that this spending sustains the forecast volumes.

The final check confirms that each Q4 2026 selection lies inside EOG's printed Q4 range.

In [ ]:
if forecast_inputs_ready:
    try:
        history_refs = assumptions_module.history_references(
            annual=annual, revenue_disclosures=revenue_disclosures, revenue_bridge=revenue_bridge,
            cost_bridge=cost_bridge, cost_unit_metrics=cost_unit_metrics, cashflow_review=cashflow_review,
            aro_settled=aro_settled, steo_annual=steo_annual)
        assumption_values, assumptions_table = assumptions_module.resolve_assumptions(
            selections, refs=history_refs, guidance=guidance, steo_annual=steo_annual,
            steo_stub=steo_stub, steo_release=steo_info["release"])
        display(Markdown("**Historical references in forecast units (FY history; prices vs EIA annual spot averages)**"))
        display(history_refs.T.round(2))
        display(Markdown("**Selected assumptions by period** (growth and inflation in %, prices per unit, amounts in USD millions)"))
        compact = assumptions_table.pivot(index="driver", columns="forecast period", values="selected assumption")
        display(compact.loc[[d["driver"] for d in selections["drivers"]]].round(3))
        display(Markdown("**Full assumptions table**"))
        with pd.option_context("display.max_colwidth", 120):
            display(assumptions_table.drop(columns=["label"]))
        guidance_checks = assumptions_module.validate_against_guidance(assumption_values, guidance, BASE_YEAR, STUB_QUARTER)
        display(guidance_checks)
        print(assumptions_table.status.value_counts().to_string())
    except (assumptions_module.AssumptionError, KeyError, ValueError) as error:
        forecast_inputs_ready = False
        print(f"Assumptions incomplete: {error}")

## 13. Operating forecast and unlevered free cash flow

The forecast reuses the historical formulas:

- Oil and NGL: `MBbl/day x days / 1,000 x USD/Bbl`; gas: `MMcf/day x days / 1,000 x USD/Mcf`, priced separately for U.S. and international gas.
- Cash costs per Boe x production (6 Mcf = 1 Boe for cost intensity only); taxes other than income as % of upstream sales.
- **Marketing is net**: third-party marketing revenue and its purchase cost are left out together; only the historical net margin is kept.
- Derivative gains and asset-sale gains are **not forecast**.

Then:

`UFCF = EBIT x (1 - tax rate) + DD&A + recurring impairments + deferred share of taxes - capex for UFCF - change in operating NWC - asset-retirement settlements`

- **Capex for UFCF** = EOG's guidance-basis capex less capitalized interest (a financing cost, already in the cost of debt) less dry-hole cost (already expensed in EBIT).
- **Recurring impairments** reduce EBIT but are noncash; the related cash (leasehold) is inside capex, so they are added back. Large price-driven impairments are not forecast.
- Stock compensation stays inside G&A as a real cost; it is **not** added back.
- The deferred share of taxes is added back in explicit years only; the terminal value is fully cash-taxed.

The FY2026E bridge shows nine reported months + Q4 forecast and checks the result against FY2026 guidance ranges.

In [ ]:
forecast_ready = False
if forecast_inputs_ready:
    try:
        forecast_statements, forecast_periods, forecast_warnings = forecast_module.build_forecast(
            assumption_values, base_year=BASE_YEAR, stub_quarter=STUB_QUARTER, years=FORECAST_YEARS,
            nine_month=nine_month, balance=latest_balance)
        display(Markdown("**Forecast statements and UFCF (USD millions unless the row says otherwise)**"))
        display(forecast_statements.round(1))
        fy_ranges = {}
        for metric in ("oil_mbbl_day", "ngl_mbbl_day", "gas_mmcf_day", "total_mboe_day", "capex_nongaap"):
            rows = guidance[(guidance.metric == metric) & (guidance.period == f"{BASE_YEAR}FY") & guidance.low.notna()]
            if not rows.empty:
                fy_ranges[metric] = (rows.iloc[0].low, rows.iloc[0].high)
        display(Markdown(f"**FY{BASE_YEAR}E = nine months reported + Q{STUB_QUARTER} forecast**"))
        display(forecast_module.fy_base_bridge(nine_month, forecast_periods[f"{BASE_YEAR}Q{STUB_QUARTER}E"],
                                               base_year=BASE_YEAR, guidance_ranges=fy_ranges).round(1))
        ufcf_identity = pd.Series({name: p["ufcf"] - (p["nopat"] + p["dd_a"] + p["impairments"] + p["deferred_tax"]
                                                     - p["capex_ufcf"] - p["change_nwc"] - p["aro"])
                                   for name, p in forecast_periods.items()})
        print("UFCF identity residual (USD millions):", ufcf_identity.abs().max().round(9))
        for message in forecast_warnings:
            print("WARNING:", message)
        forecast_ready = True
    except ValueError as error:
        print(f"Forecast unavailable: {error}")

## 14. Weighted average cost of capital (WACC)

`Cost of equity = risk-free rate + beta x equity risk premium`
`After-tax cost of debt = pre-tax cost of debt x (1 - tax rate)`
`WACC = equity weight x cost of equity + debt weight x after-tax cost of debt`

| Input | Choice | Why |
|---|---|---|
| Risk-free | 10-year Treasury par yield on the valuation date | Matches long-dated cash flows |
| Beta | Damodaran E&P unlevered beta, relevered at EOG's market D/E | A single-stock regression beta (FMP shows one) is noisy; an industry beta is more stable |
| ERP | Ten-year average of Damodaran's implied ERP | Smooths year-to-year swings; the latest value is shown too |
| Cost of debt | (guided net interest + capitalized interest) / average debt | Uses EOG's own borrowing cost |
| Weights | Market equity (price x diluted shares) and book debt | Debt includes finance leases, not operating leases |

Every input is listed with its source and status. Rules can be changed in the `wacc` block of the assumptions file.

In [ ]:
wacc_ready = False
if forecast_ready:
    try:
        prior_year_end = f"{BASE_YEAR - 1}-12-31"
        wacc_resolved, wacc_sources = workflow_module.wacc_inputs(
            selections, values=assumption_values, years=FORECAST_YEARS, treasury=treasury, erp=erp_reference,
            industry=beta_reference, profile=price_snapshot, cover=cover, balance=latest_balance,
            nine_month_shares={"diluted": quarterly.at[f"{BASE_YEAR}Q{STUB_QUARTER - 1}", "diluted_shares"],
                               "basic": quarterly.at[f"{BASE_YEAR}Q{STUB_QUARTER - 1}", "basic_shares"]},
            guidance_interest={"net_interest": guidance_module.guidance_value(guidance, "net_interest", f"{BASE_YEAR}FY"),
                               "capitalized_interest": guidance_module.guidance_value(guidance, "capitalized_interest", f"{BASE_YEAR}FY")},
            prior_year_end_debt=quarter_balances.loc[prior_year_end, "debt_current"] + quarter_balances.loc[prior_year_end, "debt_noncurrent"])
        with pd.option_context("display.max_colwidth", 140):
            display(wacc_sources)
        wacc_check = valuation_module.build_wacc(
            risk_free=wacc_resolved["risk_free"], unlevered_beta=wacc_resolved["unlevered_beta"],
            equity_risk_premium=wacc_resolved["equity_risk_premium"], pre_tax_cost_of_debt=wacc_resolved["pre_tax_cost_of_debt"],
            tax_rate=wacc_resolved["tax_rate"], share_price=wacc_resolved["share_price"],
            diluted_shares=wacc_resolved["diluted_shares"], debt=wacc_resolved["debt"])
        display(wacc_check.table().style.format("{:.4f}"))
        wacc_ready = True
    except (valuation_module.ValuationError, KeyError, TypeError) as error:
        print(f"WACC unavailable: {error}")

## 15. Terminal value, enterprise value and implied share price

**Discounting.** Cash flows are discounted from the valuation date with the **mid-period convention** (each period's cash arrives in its middle). The Q4 2026 cash flow is prorated to the days after the valuation date.

**Two terminal values**, both at the end of FY2031:
- **Perpetuity growth**: `final-year UFCF (fully cash-taxed) x (1 + g) / (WACC - g)`, adjusted by half a year because flows are mid-year. Requires WACC > g.
- **Exit multiple**: `FY2031 EBITDA x EV/EBITDA multiple`. The default multiple is EOG's **current** EV / last-twelve-months EBITDA, calculated below from the share price and the quarterly tables. It is market-implied, so treat it as a cross-check rather than independent evidence.

Each method also shows what it implies about the other (implied exit multiple, implied perpetual growth).

**Enterprise value to equity value**: subtract debt, add cash (both at 2026-09-30), add the Woodford sale proceeds received after that date, subtract dividends already declared and unpaid. No preferred stock or noncontrolling interest is reported. Operating leases are not subtracted because their cost is already in operating expenses. Cash generated and shares bought back between October 1 and the valuation date are not captured, a known approximation.

In [ ]:
valuation_ready = False
if wacc_ready:
    try:
        last_q = f"{BASE_YEAR}Q{STUB_QUARTER - 1}"
        prior_q4 = f"{BASE_YEAR - 1}Q4"
        ltm_ebitda = sum(quarterly.at[p, "reported_operating_income"] + quarterly.at[p, "dd_a"] for p in (prior_q4, f"{BASE_YEAR}YTD"))
        market_ev = wacc_resolved["share_price"] * wacc_resolved["diluted_shares"] + wacc_resolved["debt"] - latest_balance["cash"]
        terminal = selections["terminal"]
        exit_spec = terminal["exit_multiple"]
        EXIT_MULTIPLE = market_ev / ltm_ebitda if exit_spec.get("rule") == "current_ev_ltm_ebitda" else float(exit_spec["value"])
        TERMINAL_GROWTH = terminal["perpetuity_growth"]["value"] / 100
        print(f"Current EV {market_ev:,.0f} / LTM EBITDA {ltm_ebitda:,.0f} ({prior_q4} + 9M {BASE_YEAR}) = {market_ev / ltm_ebitda:.2f}x")

        divestiture_id = selections["equity_bridge"]["include_post_balance_sheet_divestiture"]
        divestiture = guidance.set_index("id").loc[divestiture_id]
        bridge_adjustments = [
            (f"Plus: post-balance-sheet sale proceeds ({divestiture_id}, before closing adjustments)", divestiture.point),
            ("Less: dividends payable at the balance-sheet date", -latest_balance["dividends_payable"]),
        ]
        valuation_kwargs = dict(nine_month=nine_month, balance=latest_balance, inputs=wacc_resolved,
                                bridge_adjustments=bridge_adjustments)
        base_case = workflow_module.run_valuation(assumption_values, selections, years=FORECAST_YEARS,
                                                  growth=TERMINAL_GROWTH, exit_multiple=EXIT_MULTIPLE, **valuation_kwargs)
        display(Markdown(f"**Discount schedule** (WACC {base_case['rate']:.2%}; Q4 stub fraction {base_case['stub_fraction']:.3f})"))
        display(base_case["schedule"].round(4))
        dcf_result = base_case["dcf"]
        summary = pd.DataFrame({
            "Perpetuity growth": {"Terminal assumption": f"g = {TERMINAL_GROWTH:.2%}",
                                  "Terminal value": dcf_result["perpetuity"]["terminal_value"],
                                  "PV of explicit cash flows": dcf_result["pv_explicit"],
                                  "PV of terminal value": dcf_result["perpetuity"]["pv_terminal"],
                                  "Enterprise value": dcf_result["perpetuity"]["enterprise_value"],
                                  "Terminal value share of EV": dcf_result["perpetuity"]["pv_terminal"] / dcf_result["perpetuity"]["enterprise_value"],
                                  "Cross-check": f"implied exit multiple {dcf_result['perpetuity']['implied_exit_multiple']:.2f}x",
                                  "Implied value per share": base_case["price_perpetuity"]},
            "Exit multiple": {"Terminal assumption": f"{EXIT_MULTIPLE:.2f}x EBITDA",
                              "Terminal value": dcf_result["exit_multiple"]["terminal_value"],
                              "PV of explicit cash flows": dcf_result["pv_explicit"],
                              "PV of terminal value": dcf_result["exit_multiple"]["pv_terminal"],
                              "Enterprise value": dcf_result["exit_multiple"]["enterprise_value"],
                              "Terminal value share of EV": dcf_result["exit_multiple"]["pv_terminal"] / dcf_result["exit_multiple"]["enterprise_value"],
                              "Cross-check": f"implied perpetual growth {dcf_result['exit_multiple']['implied_growth']:.2%}",
                              "Implied value per share": base_case["price_exit_multiple"]},
        })
        display(summary)
        display(Markdown("**Enterprise-to-equity bridge (perpetuity method; USD millions)**"))
        display(base_case["bridge_perpetuity"].round(2))
        print(f"Market price ${wacc_resolved['share_price']:.2f} (FMP snapshot). Implied: "
              f"${base_case['price_perpetuity']:.2f} perpetuity, ${base_case['price_exit_multiple']:.2f} exit multiple.")
        valuation_ready = True
    except (valuation_module.ValuationError, KeyError, ValueError) as error:
        print(f"Valuation unavailable: {error}")

## 16. Scenarios, sensitivities and model warnings

**Scenarios** (Bear / Base / Bull / Custom) are defined in the `scenarios` block of the assumptions file as changes to the Base inputs: WTI and Henry Hub shifts for FY2027 onward, a change in annual volume growth, extra cost inflation, and (Custom only) WACC and terminal-growth shifts. Every scenario runs through the **same** forecast and valuation functions; only inputs differ. Edit Custom to test your own view.

**Sensitivities** recompute the implied share price on a grid:
- WACC x terminal growth (perpetuity method),
- WACC x exit multiple,
- a flat WTI for FY2027 onward x WACC.

Limitation: in the WTI grid capex per Boe does not fall with lower prices, although EOG would likely cut activity; low-price values are therefore conservative. Cells where WACC <= g are blank.

In [ ]:
if valuation_ready:
    scenario_rows = []
    for name, deltas in selections["scenarios"].items():
        try:
            result = workflow_module.run_valuation(assumption_values, selections, years=FORECAST_YEARS,
                                                   growth=TERMINAL_GROWTH, exit_multiple=EXIT_MULTIPLE,
                                                   scenario=deltas, **valuation_kwargs)
            final = result["periods"][f"FY{FORECAST_YEARS[-1]}E"]
            scenario_rows.append({"Scenario": name, **{k: v for k, v in deltas.items()},
                                  "WACC": result["rate"], f"FY{FORECAST_YEARS[0]}E UFCF": result["periods"][f"FY{FORECAST_YEARS[0]}E"]["ufcf"],
                                  f"FY{FORECAST_YEARS[-1]}E UFCF": final["ufcf"],
                                  "Value/share, perpetuity": result["price_perpetuity"],
                                  "Value/share, exit multiple": result["price_exit_multiple"]})
        except valuation_module.ValuationError as error:
            scenario_rows.append({"Scenario": name, "Error": str(error)})
    display(pd.DataFrame(scenario_rows).set_index("Scenario").round(3))

    grids = workflow_module.sensitivities(assumption_values, selections, years=FORECAST_YEARS,
                                          base_kwargs=valuation_kwargs, base_rate=base_case["rate"],
                                          growth=TERMINAL_GROWTH, exit_multiple=EXIT_MULTIPLE)
    for title, grid in grids.items():
        display(Markdown(f"**{title}**"))
        grid = grid.copy()
        grid.index = [f"{x:.2%}" if x < 1 else f"{x:,.2f}" for x in grid.index]
        grid.columns = [f"{x:.2%}" if x < 1 else f"{x:,.2f}" for x in grid.columns]
        display(grid.round(2))

    warnings_list = list(base_case["warnings"])
    if (guidance_checks.Status != "Inside guidance range").any():
        warnings_list.append("A Q4 selection is outside EOG's printed guidance range.")
    if (quarterly_checks.Status != "Ties within rounding").any():
        warnings_list.append("Quarterly tables do not tie to year-to-date totals.")
    share_tv = dcf_result["perpetuity"]["pv_terminal"] / dcf_result["perpetuity"]["enterprise_value"]
    if share_tv > 0.75:
        warnings_list.append(f"Terminal value is {share_tv:.0%} of perpetuity EV: value depends mostly on the long-run assumptions.")
    warnings_list += [
        "Guidance values for Q4/FY2026 were transcribed from a filed image (hash-checked, not machine-read).",
        "FY2027+ volumes, long-term prices, cost inflation and terminal growth are analyst assumptions.",
        "Share price is an FMP snapshot at retrieval time; refresh before relying on the comparison.",
        "Equity bridge uses 2026-09-30 debt and cash; October cash flow and buybacks are not captured.",
    ]
    display(Markdown("**Model warnings**"))
    for message in warnings_list:
        print("-", message)

### What this model does and does not establish

- It turns sourced history, filed guidance and an explicit price deck into a five-year UFCF forecast and two terminal values. Every input is traceable in Sections 12-15.
- It is **not** a reserve-based (NAV) model: volumes after 2026 follow assumed growth with capex scaled to production, not a drilling inventory or decline curve. Reserves are not yet in the workflow.
- Results are most sensitive to the long-term oil price, WACC and the terminal assumptions; read the Section 16 grids before quoting a single value.
- When EOG publishes Q4 2026 results and 2027 guidance, add the new records to `data/reference/eog_guidance.json`, update the dates in the assumptions file, refresh the caches, and rerun.

## Refresh next year

The year range advances when a new annual period appears in a refreshed source. It does not advance merely because the calendar changes. To display five years and calculate the earliest change, retrieve six annual records.

**WRDS:** In a separate PowerShell window at the project root, run:

```powershell
.\.venv-wrds\Scripts\python.exe -m app.refresh_wrds --ticker EOG --years 6
```

Sign in through the WRDS prompts and approve Duo. Then rerun this notebook with `FINANCIAL_SOURCE = "wrds"`.

**SEC:** Add `SEC_USER_AGENT=DCF-Model your-contact-email` to your existing local `.env`, replacing the contact text with your email. Do not overwrite the existing file or paste credentials into the notebook. Set `FINANCIAL_SOURCE = "sec"` and `REFRESH_SEC = True`, then run the first cell. After a successful download set `REFRESH_SEC = False` for cached exploration. No SEC API key is needed. The SEC request includes your configured contact header.

SEC's [API documentation](https://www.sec.gov/search-filings/edgar-application-programming-interfaces) explains the standard concepts it exposes; custom company disclosures need a separate adapter. This notebook's SEC mapping is conservative, so unsupported tags remain blank and visible in the coverage table.

The original root-level `EOG_Financials.ipynb` remains your prior static research notebook. Continue new dynamic work here to avoid reintroducing its hardcoded arrays.

**Forward model (Sections 12-16).** After a new earnings release:
1. Set `REFRESH_SEC = True` in the setup cell and run it, so the new 8-K and 10-Q appear in the SEC metadata. Set it back to `False`.
2. Add the new guidance rows and the new supplemental exhibit to `data/reference/eog_guidance.json` (accession, document name, and for an image page its SHA-256), then set `REFRESH_GUIDANCE = True` once.
3. Update `valuation_date`, `balance_sheet_date`, `base_year` and `stub_quarter` in `data/assumptions/EOG_assumptions.json`.
4. Set `REFRESH_EIA = True` and `REFRESH_MARKET = True` once for a new STEO and market snapshot, then set all flags back to `False`.

The forward cells need `openpyxl` and `xlrd` (listed in `requirements-notebook.txt`).